#Graded Challenge 3

Nama  : Khoizuron Syakira\
Batch : CODA-023-RMT\

Program ini dibuat untuk melakukan web scraping data seblak di Tokopedia 
dan membersihkan datanya untuk analisis penjualan.

In [ ]:
import time as t
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup
import re

driver = webdriver.Chrome()


In [ ]:
# link
url = 'https://www.tokopedia.com/search?st=&q=seblak&srp_component_id=02.01.00.00&srp_page_id=&srp_page_title=&navsource='

# 1. Buka URL Tokopedia
driver.get(url)
t.sleep(3)

# 2. Ambil tinggi layar (screen height) sesuai metode dari artikel Medium 'https://medium.com/analytics-vidhya/using-python-and-selenium-to-scrape-infinite-scroll-web-pages-825d12c24ec7'
screen_height = driver.execute_script("return window.screen.height;")
i = 1

# 3. Loop scroll bertahap per layar
while True:
  # Scroll sejauh kelipatan tinggi layar
  driver.execute_script(f"window.scrollTo(0, {screen_height * i});")
  i += 1
  t.sleep(3)  # Jeda agar elemen sempat ter-render

  # Update total tinggi halaman
  scroll_height = driver.execute_script("return document.body.scrollHeight;")

  # Berhenti jika posisi scroll sudah melampaui total tinggi halaman
  if screen_height * i > scroll_height:
    break

# 4. Ambil dan parse HTML setelah seluruh produk termuat
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

with open('seblak.html', 'w', encoding='utf-8') as f:
    f.write(driver.page_source)

In [ ]:
# Uji coba ambil elemen nama produk
el_names = soup.find_all('span', {'class': '+tnoqZhn89+NHUA43BpiJg=='})

names = []
for name in el_names:
    names.append(name.get_text(strip=True))

names

In [ ]:
# Uji coba ambil elemen harga produk
el_prices = soup.find_all('div', {'class': 'urMOIDHH7I0Iy1Dv2oFaNw=='})

prices = []
for price in el_prices:
  prices.append(price.get_text(strip=True))

prices

In [ ]:
# Uji coba ambil elemen nama toko
el_shops = soup.find_all('span', {'class': 'si3CNdiG8AR0EaXvf6bFbQ== gxi+fsEljOjqhjSKqjE+sw== flip'})

shops = []
for shop in el_shops:
    shops.append(shop.get_text(strip=True))

shops

In [ ]:
# Uji coba ambil elemen kota
el_cities = soup.find_all('span', {'class': 'gxi+fsEljOjqhjSKqjE+sw== flip'})

cities = []
for city in el_cities:
    cities.append(city.get_text(strip=True))

cities

In [ ]:
# Uji coba ambil elemen terbanyak terjual
el_solds = soup.find_all('span', {'class': 'u6SfjDD2WiBlNW7zHmzRhQ=='})

solds = []
for sold in el_solds:
    solds.append(sold.get_text(strip=True))

solds

In [ ]:
# Uji coba ambil elemen rating
el_ratings = soup.find_all('span', {'class': '_2NfJxPu4JC-55aCJ8bEsyw=='})

ratings = []
for rating in el_ratings:
    ratings.append(rating.get_text(strip=True))

ratings

In [ ]:
# 3. Cari elemen judul (atau div produk)
title_elements = soup.find_all('span', {'class': '+tnoqZhn89+NHUA43BpiJg=='})

print(f'Total produk terdeteksi: {len(title_elements)}')

# 4. Jalankan loop jika total produk > 0
names, prices, shops, cities, solds, ratings = [], [], [], [], [], []


for title_el in title_elements:
  box = title_el.parent.parent.parent.parent

  # 1. Nama Produk
  try:
    names.append(title_el.get_text(strip=True))
  except:
    names.append(None)

  # 2. Harga Produk
  try:
    teks_kartu = box.get_text(' ', strip=True)       # semua teks di kartu jadi satu kalimat
    cocok = re.search(r'Rp\s?[\d.]+', teks_kartu)    # cari 'Rp' + angka pertama
    prices.append(cocok.group())
  except:
    prices.append(None)

  # 3. Penjual
  try:
    sh = box.find('span', {'class': 'si3CNdiG8AR0EaXvf6bFbQ=='})
    shops.append(sh.get_text(strip=True))
  except:
    shops.append(None)

  # 4. Kota
  try:
    semua = box.find_all('span', {'class': 'gxi+fsEljOjqhjSKqjE+sw=='})
    cities.append(semua[-1].get_text(strip=True))   # ambil yang TERAKHIR
  except:
    cities.append(None)

  # 5. Banyak Terjual
  try:
    so = box.find('span', {'class': 'u6SfjDD2WiBlNW7zHmzRhQ=='})
    solds.append(so.get_text(strip=True))
  except:
    solds.append(None)

  # 6. Rating
  try:
    r = box.find('span', {'class': '_2NfJxPu4JC-55aCJ8bEsyw=='})
    ratings.append(r.get_text(strip=True))
  except:
    ratings.append(None)

# Tutup browser
driver.quit()

In [ ]:
df_kotor = pd.DataFrame({
    'nama_produk': names,
    'harga_produk': prices,
    'penjual': shops,
    'kota_toko': cities,
    'banyak_terjual': solds,
    'rating_produk': ratings
})

df_kotor.to_csv('seblak_kotor.csv', index=False)

print(df_kotor['harga_produk'].isnull().sum())
print(df_kotor.shape)

In [ ]:
# Membaca data kotor dari CSV untuk diolah
df = pd.read_csv('seblak_kotor.csv')

#menghapus data missing values 
df = df.dropna(subset=['harga_produk'])

# Mengecek dimensi (jumlah baris, jumlah kolom)
df.shape

df.info()

df[['nama_produk', 'harga_produk']].head(10)


In [ ]:
# Cleaning harga produk
# Harga: 'Rp13.350' -> 13350
df['harga_produk'] = df['harga_produk'].str.replace('Rp', '', regex=False)   # buang 'Rp'
df['harga_produk'] = df['harga_produk'].str.replace('.', '', regex=False)    # buang titik ribuan
df['harga_produk'] = df['harga_produk'].astype(int)                          # ubah ke angka

In [ ]:
# Menghapus kata 'terjual', '+', konversi 'rb' ke '000', dan ubah tipe data ke float
df['banyak_terjual'].unique()
t = df['banyak_terjual']
t = t.str.replace(' terjual', '', regex=False)   # '100rb+ terjual' -> '100rb+'
t = t.str.replace('+', '', regex=False)          # -> '100rb'
t = t.str.replace('rb', '000', regex=False)      # -> '100000'
df['banyak_terjual'] = t.astype(float)           # float karena masih ada NaN

In [ ]:
# Mengubah tipe data rating ke float
df['rating_produk'] = df['rating_produk'].astype(float)

In [ ]:
df[df['rating_produk'].isnull()]
df[df['banyak_terjual'].isnull()]

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()       # semua kolom angka harus bertipe int/float, dan tidak ada yang kosong
df.describe()   # lihat min dan max


In [ ]:
df.to_csv('seblak_bersih.csv', index=False)